Warum nutzt BlackRock Marktanalysen mit Clustering
- verschiedene Marktphasen birgen unterschiedliche Risiken
- BlackRock kann Risiken früher erkennen und Portfolios und Investments anpassen


In [ ]:
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dax = yf.Ticker("^GDAXI")

# DAX von 2018 bs 2024
dax = yf.download("^GDAXI", start="2018-01-01", end="2024-12-31")

In [ ]:
# Rendite berechnen
# tägliche Prozentuale Veränderung
dax["Rendite"] = dax["Adj Close"].pct_change()

In [ ]:
# Volatilität berechnen
dax["Volatilität"] = dax["Rendite"].rolling(window=20).std()

In [ ]:
# Dataset bereinigen
dax = dax.dropna()

In [ ]:
features = dax[["Rendite","Volatilität" ]]

In [ ]:
# Gewichtung der Features gleichwertig machen 
scaler = StandardScaler()
X = np.array(features)
X_scaled = scaler.fit_transform(X)

In [ ]:
# Modell
# K_means
# K = 3 (ruhig, mittel, volatil)

kmeans = KMeans(n_clusters=3, random_state=0, n_init="auto").fit(X_scaled)
dax["Cluster"] = kmeans.labels_

In [ ]:

plt.figure()
plt.scatter(features["Rendite"], features["Volatilität"])
plt.xlabel("Rendite")
plt.ylabel("Volatilität")
plt.title("Marktdaten ohne Clustering")
plt.show()

In [ ]:
labels = kmeans.labels_

plt.figure()
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels)
plt.xlabel("Rendite (skaliert)")
plt.ylabel("Volatilität (skaliert)")
plt.title("Clustering der Marktphasen")
plt.show()

In [ ]:
# Letzte Marktphase einordnen
latestDax = yf.download("^GDAXI", start="2025-01-01", end="2025-12-31")

latestDax["Rendite"] = latestDax["Adj Close"].pct_change()
latestDax["Volatilität"] = latestDax["Rendite"].rolling(window=20).std()

newFeatures = latestDax[["Rendite","Volatilität" ]]

features_scaled = scaler.transform(newFeatures)

kmeans.predict(features_scaled)
